# Flat array manipulation
Up to this point we have been asuming that we work with fixed size arrays, reshaped to the appropiate size. However, it is usually a good idea to work a flat array and retrieve and assign matrix elements by index using a function (or macro). 

That way, it is possible to introduce a fixed size array, and manipulate it in order to not allocate memory. 

First lets consider a matrix: 

In [1]:
import numpy as np 
from numpy.typing import NDArray

In [2]:
flat_array = np.linspace(0, 35, 36)

In [3]:
flat_array

array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.,
       13., 14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.,
       26., 27., 28., 29., 30., 31., 32., 33., 34., 35.])

In [4]:
array = flat_array.reshape([6,6])
array

array([[ 0.,  1.,  2.,  3.,  4.,  5.],
       [ 6.,  7.,  8.,  9., 10., 11.],
       [12., 13., 14., 15., 16., 17.],
       [18., 19., 20., 21., 22., 23.],
       [24., 25., 26., 27., 28., 29.],
       [30., 31., 32., 33., 34., 35.]])

Which shows that the arrays are row ordered. Therefore we will define our ordering with row as the first index, column as the second, ... We can define a function:

In [5]:
def get_index(n, m, n_size, m_size) -> int: 
    return n * n_size + m

For example in our case the element `27` is assigned with `[4,3]`:

In [6]:
array[4,3]

np.float64(27.0)

Or we could use: 

In [7]:
flat_array[get_index(4,3,6,6)]

np.float64(27.0)

And here, the size of the flar array does not matter as long as the contents and boundaries are well-defined. For example if the array was not flat, but longer:

In [8]:
flat_array2 = np.linspace(0, 135, 136)
flat_array2

array([  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
        11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
        22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
        33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
        44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
        55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
        66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
        77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
        88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
        99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
       110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
       121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131.,
       132., 133., 134., 135.])

But since the order remains the same: 


In [9]:
flat_array2[get_index(4,3,6,6)]

np.float64(27.0)

Which allows passing unknown size arrays (provided they are large enough), and to manipulate them. Now we will generalize to the order of dimensions that we might encounter: 

In [10]:
def get_index_2d_C(i, j, i_size, j_size) -> int: 
    return i * j_size + j

def get_index_3d_C(i,j,k,i_size,j_size,k_size) -> int:
    return i * j_size * k_size +  j*k_size  + k 

def get_index_4D_C(i,j,k,l, i_s, j_s, k_s, l_s) -> int: 
    return i * j_s * k_s * l_s +  j * k_s * l_s + k * l_s +  l  

def get_index_5D_C(i,j,k,l,m, i_s, j_s, k_s, l_s, m_s) -> int: 
    return i * j_s * k_s * l_s * m_s +  j * k_s * l_s * m_s + k * l_s * m_s+  l * m_s + m 

Which shows that in C, it is basically to multiply each index by the max size of all the indices after it: 

$$ \text{IDX}(d_1, d_2, \dots, d_N) = \sum_{i=1}^N \left( d_i \prod_{j=i+1}^N s_j \right) $$

Where `d_i` is the integer index value of the `i`-th dimension and `s_i` is the size of the `i`-th dimension. Therefore we could build a generalized function: 


In [38]:
def idx_nd_C(indices: NDArray[int], sizes: NDArray[int], ndim: int): 
    assert (len(indices) == len(sizes) == ndim)

    final_index = 0

    for i in range(ndim-1): 
        product = 1
        for j in range(i+1, ndim):
            product *= sizes[j]
        final_index += indices[i] * product
    final_index += indices[-1]
    
    return final_index 

In [39]:
array_3d = flat_array.reshape([6,3,2])
array_3d

array([[[ 0.,  1.],
        [ 2.,  3.],
        [ 4.,  5.]],

       [[ 6.,  7.],
        [ 8.,  9.],
        [10., 11.]],

       [[12., 13.],
        [14., 15.],
        [16., 17.]],

       [[18., 19.],
        [20., 21.],
        [22., 23.]],

       [[24., 25.],
        [26., 27.],
        [28., 29.]],

       [[30., 31.],
        [32., 33.],
        [34., 35.]]])

In [40]:
print(array_3d[1,0,0])
print(array_3d[1,1,1])
print(array_3d[2,0,0])


6.0
9.0
12.0


In [41]:
print(flat_array[get_index_3d_C(1,0,0,6,3,2)])
print(flat_array[get_index_3d_C(1,1,1,6,3,2)])
print(flat_array[get_index_3d_C(2,0,0,6,3,2)])

6.0
9.0
12.0


In [42]:
print(flat_array.reshape([3,3,2,2])[0,1,0,1])

5.0


In [43]:
flat_array[get_index_4D_C(0,1,0,1,3,3,2,2)]

np.float64(5.0)

In [44]:
flat_array2 = np.linspace(0,215,216)
flat_array2

array([  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
        11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
        22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
        33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
        44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
        55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
        66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
        77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
        88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
        99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
       110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
       121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131.,
       132., 133., 134., 135., 136., 137., 138., 139., 140., 141., 142.,
       143., 144., 145., 146., 147., 148., 149., 15

In [45]:
print(flat_array2.reshape([6,3,3,2,2])[4,2,2,1,0])
print(flat_array2[get_index_5D_C(4,2,2,1,0,6,3,3,2,2)])

178.0
178.0


Or:

In [46]:
indices = [4,2,2,1,0]
sizes = [6,3,3,2,2]
n_dim = 5 

flat_array2[idx_nd_C(indices, sizes, n_dim)]


np.float64(178.0)